In [ ]:
# Import dependencies

from PIL import Image
import os
import json
import re
from textwrap import dedent
import pandas as pd
# My specific setting.
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/LlavaGuard')
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph
from pegiguard.data.prompts import build_prompt
from llavaguard_config import local_data_dir


In [ ]:
# Correct workspace image paths

def fix_image_path(img_path: str) -> str:
    return img_path.replace(
        "/mlde_wsp_KIServiceCenter/",
        "/mlde_wsp_PI_Kersting/",
    )

file_path = f"{local_data_dir}/labels/llama4_v1/unsafebench/unsafebench.csv"
df = pd.read_csv(file_path)
df["image_path"] = df["image_path"].apply(fix_image_path)
df.to_csv(file_path, index=False)


In [ ]:
# Compare annotation files

df_1_path = f"{local_data_dir}/labels/qwen3_5/llavaguard/llavaguard.csv"
df_2_path = f"{local_data_dir}/labels/gemma/llavaguard.csv"

df_1 = pd.read_csv(df_1_path)
df_2 = pd.read_csv(df_2_path)

# Find the extra line in gemma.
keys1 = set(zip(df_1['image_path'], df_1['subcategory'], df_1['rating']))
keys2 = set(zip(df_2['image_path'], df_2['subcategory'], df_2['rating']))

extra_in_df1 = keys1 - keys2
extra_in_df2 = keys2 - keys1

print(f"Extra in 1: {len(extra_in_df1)}")
print(f"Extra in 2: {len(extra_in_df2)}")

if extra_in_df1:
    for key in extra_in_df1:
        row = df_1[(df_1['image_path'] == key[0]) & (df_1['subcategory'] == key[1]) & (df_1['rating'] == key[2])]
        print("Extra row in DF #1:")
        print(row)
        print("\n")

if extra_in_df2:
    for key in extra_in_df2:
        row = df_2[(df_2['image_path'] == key[0]) & (df_2['subcategory'] == key[1]) & (df_2['rating'] == key[2])]
        print("Extra row in DF #2:")
        print(row)
        
        print("\n")


In [ ]:
# Build an economic-content prompt

def build_prompt_economic(category: str, subcategory: str, graph_info):

    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase - the examples are illustrative, not exhaustive).
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt


In [ ]:
# Encode image data

from io import BytesIO
import base64

def pil_to_data_url(img: Image.Image, fmt="JPEG") -> str:
    buf = BytesIO()
    img.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    mime = "image/jpeg" if fmt.upper() == "JPEG" else f"image/{fmt.lower()}"
    return f"data:{mime};base64,{b64}"


In [ ]:
# Append annotation rows

import csv, json, os

def append_rows_to_csv(rows, csv_path):
    """Append assessment records to a CSV, creating it with a header if needed."""
    # Normalize/flatten the row structure for CSV.
    flat_rows = []
    for r in rows:
        flat_rows.append({
            "image_path": r.get("image_path"),
            "subcategory": r.get("subcategory"),
            "rating": r.get("rating"),
            "assessment": r.get("assessment"),
            "review_count": len(r.get("review", [])),
            "review_raw": json.dumps(r.get("review", []), ensure_ascii=False),
        })

    fieldnames = ["image_path", "subcategory", "rating", "assessment", "review_count", "review_raw"]
    file_exists = os.path.exists(csv_path)

    # Write (append) with header only once.
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            w.writeheader()
        w.writerows(flat_rows)


In [ ]:
# Inspect predictions for one image

# Try for each category.
# Print the results of all categories for 1 img.

from pegiguard.data.prompts import build_prompt
import json
from io import BytesIO
import base64


system_rule = (
        "You are a careful, literal safety reviewer. "
        "Output ONLY a single YES or NO."
        "No extra text, no markdown, no explanations outside of the decision."
    )

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/nudity/image_269.jpg"

categories = list(policy_graph.keys())
subcategories = []
pos_c = 0

for category, category_details in policy_graph.items():
    for subcategory, graph_info in category_details.items():
        
        
        # #print(prompt).
        # Elif category == "S2: Hate".
        
        # Elif category == "S3: Nude Content".
        
        # #print(prompt).
        # Elif category == "S4: Disturbing Content".
        
        # Elif category == "S5: Self-Harm".
        
        # Elif category == "S6: Criminal Activities".
        
        # Elif category == "S7: Regulated Substances".
        
        # Elif category == "S8: Economic Harm".
        
        # Else.
        # Continue.
        prompt = build_prompt(category, subcategory, graph_info)
        
        image = Image.open(img_path).convert("RGB")
        data_url = pil_to_data_url(image, fmt="JPEG")
        
        messages = [
            {"role": "system", "content": system_rule},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            },
        ]

        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=3,   
        )
        response = resp.choices[0].message.content.strip()

        print(subcategory)
        print(response)

print(img_path)
im = Image.open(img_path).resize((256, 256))
im.show()
if pos_c > 0:
    im = Image.open(img_path).resize((256, 256))
    im.show()


In [ ]:
# Sample images from each directory

import os, random

def sample_images_per_dir(root: str, k: int = 5):
    SEED = 10
    rng = random.Random(SEED)

    selected = []
    for dirpath, namepaths, filenames in os.walk(root):
        imgs = [
            os.path.join(dirpath, f)
            for f in filenames
            if f.lower().endswith((".jpg"))
        ]
        if not imgs:
            continue
        # Sample up to k unique images from this dir.
        n = min(k, len(imgs))
        selected.extend(rng.sample(imgs, n))
    return selected


In [ ]:
# Label sampled images

# Try for all images in one subtree.
# Save results to a file.

system_rule = (
        "You are a careful, literal safety reviewer. "
        "Output ONLY a single YES or NO."
        "No extra text, no markdown, no explanations outside of the decision."
)

version = 5
model_name = "llama_vllm"
output_file = f"/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/l_all_{version}_{model_name}.json"
csv_file = f"/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/l_all_{version}_{model_name}.csv"

os.makedirs(os.path.dirname(output_file), exist_ok=True)
if os.path.exists(output_file):
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if not isinstance(saved, list):
            saved = []
    except Exception:
        saved = []
else:
    saved = []

batch_size = 5
img_root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images"

# K random imgs from each directory of the root.
K_PER_DIR = 3
image_paths = sample_images_per_dir(img_root, K_PER_DIR)
image_paths.sort() 
total_images = len(image_paths)
print(f"Selected {total_images} image paths (≤{K_PER_DIR} per directory) from {img_root}")


buffer  = []
processed = 0

categories = list(policy_graph.keys())
subcategories = []
pos_c = 0

for idx, img_path in enumerate(image_paths, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping unreadable image: {img_path} ({e})")
        continue
    
    for category, category_details in policy_graph.items():
        for subcategory, graph_info in category_details.items():
            # Preparing a category-specific prompt.
            prompt = build_prompt(category, subcategory, graph_info)

            image = Image.open(img_path).convert("RGB")
            data_url = pil_to_data_url(image, fmt="JPEG")
            
            messages = [
                {"role": "system", "content": system_rule},
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ]},
            ]

            resp = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                max_tokens=3,   
            )
            response = resp.choices[0].message.content.strip()
            if not (response == "NO" or response == "YES"):
                print(f"Unknown answer - {response}, img - {img_path}")
                continue
            record = {
                "image_path": img_path,
                "subcategory": subcategory,
                "rating": graph_info["rating"],
                "assessment": response,
                "review": []
            }
            buffer.append(record)

    processed += 1                 
    # Writing in json in batches.
    if processed % batch_size == 0:
        saved.extend(buffer)
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(saved, f, ensure_ascii=False, indent=2)
        append_rows_to_csv(buffer, csv_file)
        print(f"Batch OK — appended {len(buffer)} positives. Total saved: {len(saved)}")
        buffer = []
        print(f"Progress: {processed}/{total_images} images processed.")
        print(f"Saved json in {output_file}")
        print(f"Saved cvs in {csv_file}")


In [ ]:
# Check sampled image paths

# Checking if the selected paths are valid.
img_root = f"{local_data_dir}/images"

K_PER_DIR = 3
image_paths = sample_images_per_dir(img_root, K_PER_DIR)
image_paths.sort() 

total_images = len(image_paths)

print(f"Selected {total_images} image paths (≤{K_PER_DIR} per directory) from {img_root}")
counter = 0

for idx, img_path in enumerate(image_paths, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping unreadable image: {img_path} ({e})")
        continue
    print(img_path)
    counter = counter + 1
print (f"Conclusion: {counter} valid image paths")


In [ ]:
# Sample YES and NO examples

import csv
import pandas as pd
from llavaguard_config import local_image_dirs, local_data_dir

csv_path = f"{local_data_dir}/labels/pegi/v2/labels_cleaned_v1.csv"

output_path = f"{local_data_dir}/labels/pegi/v2/eval_positive_v2.csv"

df = pd.read_csv(csv_path)
SEED = 42
selected = []
for subcat, g in df.groupby("subcategory", dropna=False):
    counts = g["assessment"].value_counts()
    if counts.get("YES", 0) >= 10:
        part_yes = g[g["assessment"] == "YES"].sample(n=10, random_state=SEED)
        selected.append(part_yes)
    else:
        skipped_cat = subcat
eval_df = (
    pd.concat(selected, ignore_index=True) if selected else pd.DataFrame(columns=df.columns)
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
eval_df.to_csv(output_path, index=False)

print(f"Saved {len(eval_df)} rows to {output_path} ")
print(f"Skipped subcategories (insufficient YES to take 10 each): {skipped_cat}")


In [ ]:
# Inspect selected annotation rows

# Show rows of a specific category.
def show_eval_rows():
    eval_path = f"{local_data_dir}/labels/pegi/v3/educaltional_content_v2.csv"
    df = pd.read_csv(eval_path)
    df_mask = df["assessment"] == "YES"
    for _, row in df[df_mask].iterrows():
        img_path = row["image_path"]
        subcategory = row["subcategory"]
        assessment = row["assessment"]
        review = row["review"]
        print(f"subcategory: {subcategory}")
        print(f"assessment: {assessment}")
        print(f"review: {review}")
        print(f"image_path: {img_path}")
        img = Image.open(img_path)
        img.show()


In [ ]:
# Display annotation examples

show_eval_rows()


In [ ]:
# Download UnsafeBench images

# Saves the images from unsafe bench to local files.
from datasets import load_dataset
import os 
from PIL import Image

root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/UnsafeBench"
save_root = os.path.join(root, "images_local")  
os.makedirs(save_root, exist_ok=True)
ds = load_dataset("yiting/UnsafeBench", cache_dir=root)

for split_name, split in ds.items():
    out_dir = os.path.join(save_root, split_name)  
    os.makedirs(out_dir, exist_ok=True)
    for i, ex in enumerate(split):
        img = ex["image"]        
        path = os.path.join(out_dir, f"{i}.jpg")
        try:
            img.convert("RGB").save(path, "JPEG")
        except Exception as e:
            print(f"Failed {split_name}[{i}] -> {e}")


In [ ]:
# Compare image vectors

# Check if 2 images are identical.
import numpy as np
from numpy import dot
from PIL import Image
from numpy.linalg import norm

img_path1 = f"{local_data_dir}/images/real_images/Obscene gestures/image_36.jpg"
img_path2 = f"{local_data_dir}/images/crawled_data/Obscene gestures/image_309.jpg"


a = Image.open(img_path1).convert("RGB")
b = Image.open(img_path2).convert("RGB")

if a.size != b.size:
    b = b.resize(a.size, resample=Image.BILINEAR)

# To float arrays in [0,1].
a_np = np.asarray(a, dtype=np.float32) / 255.0
b_np = np.asarray(b, dtype=np.float32) / 255.0

# Flatten.
a_vec = a_np.ravel()
b_vec = b_np.ravel()

# Cosine similarity (with tiny epsilon for safety).
den = (np.linalg.norm(a_vec) * np.linalg.norm(b_vec))
cos_sim = float(np.dot(a_vec, b_vec) / (den + 1e-12))

print(f"Cosine similarity: {cos_sim:.6f}")
if np.isclose(cos_sim, 1.0, atol=1e-6):
    print("Images are identical or nearly identical.")


In [ ]:
# Compare perceptual hashes

# The same but with hash.
from PIL import Image
import imagehash

hash1 = imagehash.phash(Image.open(img_path1))
print(f"Image 1 hash: {hash1}")
hash2 = imagehash.phash(Image.open(img_path2))

similarity = 1 - (hash1 - hash2) / len(hash1.hash)**2
print(f"Perceptual hash similarity: {similarity:.6f}")
if similarity == 1.0:
    print("Images are identical.")


In [ ]:
# Deduplicate CSV images

import imagehash
# Remove duplicates from a csv file by using imagehash.

from PIL import Image
import csv, os

input_csv  = f"{local_data_dir}/labels/pegi/v2/labels_cleaned_v1.csv"
output_csv = f"{local_data_dir}/labels/pegi/v2/labels_no_duplicates_v2.csv"
report_csv = f"{local_data_dir}/labels/pegi/v2/dupl_report.csv"

df = pd.read_csv(input_csv)
print(f"size of ds {len(df)}")

unique_paths = df["image_path"].unique()
print(f"Unique image paths (after unique()): {len(unique_paths)}")

hash_to_canonical = {}
path_to_canonical = {}
report_buffer = []

for img_path in unique_paths:
    if not os.path.exists(img_path):
        print(f"Missing file: {img_path}")
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            hsh = imagehash.phash(im)
    except Exception as e:
        print(f"Failed to hash {img_path}: {e}")
        path_to_canonical[img_path] = img_path
        continue

    match = hash_to_canonical.get(hsh, None)
    if match is None:
        # Image for the first time.
        hash_to_canonical[hsh] = img_path
        path_to_canonical[img_path] = img_path
    else:
        
        # Image already existed, a duplicate found.
        path_to_canonical[img_path] = match
        report_buffer.append({
            "canonical_path": match,
            "duplicate_path": img_path
        })

# Write report.
with open(report_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["canonical_path", "duplicate_path"])
    w.writeheader()
    w.writerows(report_buffer)
print(f"Found {len(report_buffer)} duplicates by perceptual hash.")

# Delete duplicates from the df.
canonical_paths = set(v for v in path_to_canonical.values())
df_unique = df[df["image_path"].isin(canonical_paths)].copy()

df_unique.to_csv(output_csv, index=False)
print(f"Size of deduplicated dataset: {len(df_unique)}")
print(f"Removed {len(df) - len(df_unique)} annotation rows across {len(report_buffer)} duplicate images")


In [ ]:
# Deduplicate JSON images

import imagehash
# Remove duplicates from a json file by using imagehash.


input_json  = f"{local_data_dir}/labels/pegi/v3/pegiguard_merged.json"
output_json = f"{local_data_dir}/labels/pegi/v3/pegiguard_no_dupl.json"
report_json = f"{local_data_dir}/labels/pegi/v3/dupl_report.json"

with open(input_json, "r", encoding="utf-8") as f:
        data = json.load(f)

print(f"Total records in input: {len(data)}")
keep_mask = [True] * len(data)

hash_to_canonical = {}
path_to_canonical = {}
report_buffer = []

for i, rec in enumerate(data):
    img_path = rec.get("image_path")

    if not os.path.exists(img_path):
        print(f"Missing file: {img_path}")
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            hsh = imagehash.phash(im)
    except Exception as e:
        print(f"Failed to hash {img_path}: {e}")
        path_to_canonical[img_path] = img_path
        continue

    match = hash_to_canonical.get(hsh, None)
    if match is None:
        # Image for the first time.
        hash_to_canonical[hsh] = img_path
        path_to_canonical[img_path] = img_path
    else:
        # Image already existed, a duplicate found.
        keep_mask[i] = False
        path_to_canonical[img_path] = match
        report_buffer.append({
            "canonical_path": match,
            "duplicate_path": img_path
        })

cleaned_data = [rec for rec, keep in zip(data, keep_mask) if keep]

os.makedirs(os.path.dirname(output_json) or ".", exist_ok=True)
os.makedirs(os.path.dirname(report_json) or ".", exist_ok=True)


with open(output_json, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=4)


with open(report_json, "w", encoding="utf-8") as f:
    json.dump(report_buffer, f, ensure_ascii=False, indent=4)

print(f"Unique kept: {len(cleaned_data)}")
print(f"Duplicates found: {len(report_buffer)}")
print(f"Removed {len(data) - len(cleaned_data)} entries total.")


In [ ]:
# Remove selected annotation rows

file_path = f"{local_data_dir}/labels/llama4_v1/llavaguard/llavaguard_merged_w_edu.csv"

df = pd.read_csv(file_path)

# Delete the 'review' column.


# Filter out bad categories.
bad_cats = ["46. Child Endangerment", "47. Child Sexual Abuse"]
df = df[~df['subcategory'].isin(bad_cats)]

# Define educational categories.
edu_cats = {
    "1. Educational Content": "PEGI 3",
    "10. Educational Content": "PEGI 3",
    "15. Educational Content": "PEGI 12",
    "26. Educational Content": "PEGI 3",
    "30. Educational Content": "PEGI 3",
    "38. Educational Content": "PEGI 3",
    "44. Educational Content": "PEGI 3",
}

# Group by image_path.
grouped = df.groupby('image_path')

new_rows = []

for image_path, group in grouped:
    existing_edu = set(group['subcategory']) & set(edu_cats.keys())
    missing_edu = set(edu_cats.keys()) - existing_edu
    for cat in missing_edu:
        new_row = {
            'image_path': image_path,
            'subcategory': cat,
            'rating': edu_cats[cat],
            'assessment': ''
        }
        new_rows.append(new_row)

# Add new rows.
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df = pd.concat([df, df_new], ignore_index=True)

# Save back.
df.to_csv(file_path, index=False)

print("Processing complete.")


In [ ]:
# Remove obsolete columns

file_path = f"{local_data_dir}/labels/qwen3_5/llavaguard/llavaguard.csv"

df = pd.read_csv(file_path)
df = df.drop(columns=['rating'])
df.to_csv(file_path, index=False)


In [ ]:
# Combine labels by majority vote

from pathlib import Path

# Define file paths.
f1_path = f"{local_data_dir}/labels/llama4_v1/unsafebench/unsafebench.csv"
f2_path = f"{local_data_dir}/labels/qwen3_5/unsafebench/unsafebench.csv"
f3_path = ""  f"{local_data_dir}/labels/gemma_12/unsafebench.csv"

output_path = f"{local_data_dir}/labels/majority_vote/unsafebench_mv_12.csv"

# Read all three files.
df1 = pd.read_csv(f1_path)
df2 = pd.read_csv(f2_path)
df3 = pd.read_csv(f3_path)

# Rename assessment columns to distinguish them.
df1 = df1.rename(columns={'assessment': 'assessment_1'})
df2 = df2.rename(columns={'assessment': 'assessment_2'})
df3 = df3.rename(columns={'assessment': 'assessment_3'})

# Merge on image_path and subcategory.
merged = df1.merge(df2, on=['image_path', 'subcategory'], how='outer', validate='one_to_one')
merged = merged.merge(df3, on=['image_path', 'subcategory'], how='outer', validate='one_to_one')

# Perform majority voting.
def majority_vote(row):
    """Require at least two matching valid votes; leave ties unresolved."""
    from collections import Counter
    votes = [row.get(f'assessment_{number}') for number in (1, 2, 3)]
    counts = Counter(vote for vote in votes if isinstance(vote, str) and vote in {'YES', 'NO'})
    if not counts:
        return None
    decision, count = counts.most_common(1)[0]
    return decision if count >= 2 else None

merged['assessment'] = merged.apply(majority_vote, axis=1)

# Select and write result.
result = merged[['image_path', 'subcategory', 'assessment']]
result.to_csv(output_path, index=False)

print(f"Majority voting complete!")
print(f"Total rows: {len(result)}")
print(f"Results saved to: {output_path}")
print(f"\nSample of results:")
print(result.head(10))


In [ ]:
# Keep unanimous YES decisions

def filter_unanimous_yes(f1_path, f2_path, f3_path, output_path):
    """Filter records where all three models voted YES, save all records with assessment column."""
    import pandas as pd
    
    # Read all three files.
    df1 = pd.read_csv(f1_path)
    df2 = pd.read_csv(f2_path)
    df3 = pd.read_csv(f3_path)
    
    # Rename assessment columns to distinguish them.
    df1 = df1.rename(columns={'assessment': 'assessment_1'})
    df2 = df2.rename(columns={'assessment': 'assessment_2'})
    df3 = df3.rename(columns={'assessment': 'assessment_3'})
    
    # Merge on image_path and subcategory (outer join to get all records).
    merged = df1.merge(df2, on=['image_path', 'subcategory'], how='outer', validate='one_to_one')
    merged = merged.merge(df3, on=['image_path', 'subcategory'], how='outer', validate='one_to_one')
    
    # Determine assessment: "YES" only if all three are "YES", otherwise "NO".
    def unanimous_yes_assessment(row):
        """Return "YES" only if all three models voted "YES", otherwise "NO"."""
        assessments = [row['assessment_1'], row['assessment_2'], row['assessment_3']]
        if all(a == 'YES' for a in assessments if pd.notna(a)):
            # Check if all are "YES" AND all are not NaN.
            if all(pd.notna(a) for a in assessments):
                return 'YES'
        return 'NO'
    
    merged['assessment'] = merged.apply(unanimous_yes_assessment, axis=1)
    
    # Select and write result.
    result = merged[['image_path', 'subcategory', 'assessment']]
    result.to_csv(output_path, index=False)
    
    print(f"Filter complete!")
    print(f"Total records: {len(result)}")
    print(f"Unanimous YES records: {len(result[result['assessment'] == 'YES'])}")
    print(f"NO records: {len(result[result['assessment'] == 'NO'])}")
    print(f"Results saved to: {output_path}")


In [ ]:
# Export unanimous YES decisions

filter_unanimous_yes(
    f1_path=f"{local_data_dir}/labels/llama4_v2/llavaguard_merged_w_edu.csv",
    f2_path=f"{local_data_dir}/labels/qwen3_5/llavaguard/llavaguard.csv",
    f3_path=f"{local_data_dir}/labels/gemma_12/llavaguard.csv",
    output_path=f"{local_data_dir}/labels/majority_vote/unanimous/llavaguard.csv"
)


In [ ]:
# Keep unanimous valid decisions

def filter_unanimous_strict(f1_path, f2_path, f3_path, output_path):
    """
    Filter records where all three models unanimously agreed (all YES or all NO).
    Excludes any image-subcategory pair with mixed votes (e.g., YES, YES, NO).
    Output has columns: image_path, subcategory, assessment
    """
    import pandas as pd
    
    # Read all three files.
    df1 = pd.read_csv(f1_path)
    df2 = pd.read_csv(f2_path)
    df3 = pd.read_csv(f3_path)
    
    # Rename assessment columns to distinguish them.
    df1 = df1.rename(columns={'assessment': 'assessment_1'})
    df2 = df2.rename(columns={'assessment': 'assessment_2'})
    df3 = df3.rename(columns={'assessment': 'assessment_3'})
    
    # Merge on image_path and subcategory (inner join to get only common records).
    merged = df1.merge(df2, on=['image_path', 'subcategory'], how='inner', validate='one_to_one')
    merged = merged.merge(df3, on=['image_path', 'subcategory'], how='inner', validate='one_to_one')
    
    # Filter for unanimous consensus: all three must be the same.
    def is_unanimous(row):
        """Check if all three assessments are identical."""
        a1 = row['assessment_1']
        a2 = row['assessment_2']
        a3 = row['assessment_3']
        return isinstance(a1, str) and a1 in {'YES', 'NO'} and a1 == a2 == a3
    
    # Apply filter.
    unanimous_mask = merged.apply(is_unanimous, axis=1)
    unanimous = merged[unanimous_mask].copy()
    
    # Set assessment to the unanimous value (all three are the same).
    unanimous['assessment'] = unanimous['assessment_1']
    
    # Select and write result.
    result = unanimous[['image_path', 'subcategory', 'assessment']]
    result.to_csv(output_path, index=False)
    
    yes_count = len(result[result['assessment'] == 'YES'])
    no_count = len(result[result['assessment'] == 'NO'])
    excluded_count = len(merged) - len(result)
    
    print(f"Unanimous consensus filtering complete!")
    print(f"Total unanimous records: {len(result)}")
    print(f"  - Unanimous YES: {yes_count}")
    print(f"  - Unanimous NO: {no_count}")
    print(f"Mixed/excluded records: {excluded_count}")
    print(f"Results saved to: {output_path}")


In [ ]:
# Export unanimous valid decisions

filter_unanimous_strict(
    f1_path=f"{local_data_dir}/labels/llama4_v2/llavaguard_merged_w_edu.csv",
    f2_path=f"{local_data_dir}/labels/qwen3_5/llavaguard/llavaguard.csv",
    f3_path=f"{local_data_dir}/labels/gemma_12/llavaguard.csv",
    output_path=f"{local_data_dir}/labels/majority_vote/unanimous/llavaguard_strict.csv"
)


In [ ]:
# Define subcategory redundancy rules

from __future__ import annotations
from pathlib import Path
from typing import Dict, List

REDUNDANCY_RULES: Dict[str, List[str]] = {
    # Violence.
    "9. Gross Violence": [
        "6. Realistic Violence",
        "4. Mild Violence",
        "2. Unrealistic and Implied Violence",
    ],
    "6. Realistic Violence": [
        "4. Mild Violence",
        "2. Unrealistic and Implied Violence",
    ],
    "4. Mild Violence": [
        "2. Unrealistic and Implied Violence",
    ],
    "12. Offensive Language": [
        "11. Mild Swearing",
    ],

    # Nudity.
    "18. Explicit Sexual Content": [
        "19. Explicit Nudity Content",
        "17. Mild Nudity Content",
        "16. Sexual Innuendo",
    ],
    "19. Explicit Nudity Content": [
        "17. Mild Nudity Content",
        "16. Sexual Innuendo",
    ],
    "17. Mild Nudity Content": [
        "16. Sexual Innuendo",
    ],

    # Disturbing.
    "24. Highly Disturbing Content": [
        "22. Moderately Disturbing Content",
        "21. Scary Content",
    ],
    "22. Moderately Disturbing Content": [
        "21. Scary Content",
    ],

    # Self-harm.
    "28. Suicide": [
        "29. Self-injury",
        "27. Non-realistic Self-harm",
    ],
    "29. Self-injury": [
        "27. Non-realistic Self-harm",
    ],
}


def clean_redundant_labels(csv_path: str, output_path: str | None = None) -> pd.DataFrame:
    """
    Clean redundant labels in a majority-vote CSV and optionally save the result.

    For each image_path, if a higher-severity subcategory is assessed as YES,
    lower-severity redundant subcategories for the same image are forced to NO.
    """
    csv_path = str(csv_path)
    output_path = output_path or csv_path

    df = pd.read_csv(csv_path)

    required_cols = {"image_path", "subcategory", "assessment"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Work image by image instead of row by row.
    for image_path, group_idx in df.groupby("image_path").groups.items():
        image_rows = df.loc[group_idx]

        yes_subcategories = set(
            image_rows.loc[image_rows["assessment"] == "YES", "subcategory"]
        )

        # Collect all lower-priority labels that should be suppressed.
        subcategories_to_disable = set()
        for yes_subcat in yes_subcategories:
            subcategories_to_disable.update(REDUNDANCY_RULES.get(yes_subcat, []))

        if not subcategories_to_disable:
            continue

        mask = (
            df.index.isin(group_idx)
            & df["subcategory"].isin(subcategories_to_disable)
        )
        df.loc[mask, "assessment"] = "NO"

    df.to_csv(output_path, index=False)

    total_rows = len(df)
    yes_rows = (df["assessment"] == "YES").sum()
    no_rows = (df["assessment"] == "NO").sum()


    print("Redundant label cleaning complete")
    print(f"Total rows: {total_rows}")
    print(f"YES rows: {yes_rows}")
    print(f"NO rows: {no_rows}")
    print(f"Output saved to: {output_path}")

    return df


In [ ]:
# Clean redundant labels

clean_redundant_labels(
    csv_path=f'{local_data_dir}/labels/majority_vote/3_models/llavaguard_mv_g12.csv',
    output_path=f'{local_data_dir}/labels/majority_vote/3_models/llavaguard_mv_g12_clean.csv'
)


In [ ]:
# Count dataset records

# Calculate number of entries in the dataset.

def count_json_entries(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        return len(data)
    elif isinstance(data, dict):
        return len(data)
    else:
        raise ValueError("Unsupported JSON root type: %s" % type(data).__name__)

path_age = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/age_rating/all_data.json"
print("Count of entries in age dataset:", count_json_entries(path_age))


In [ ]:
# Compare rating and safety IDs

def load_json_list(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        data = [data]
    if not isinstance(data, list):
        raise ValueError(f"Unsupported JSON root type: {type(data).__name__}")
    return data


def compare_age_and_safety_ids(path_age, path_safety):
    age_data = load_json_list(path_age)
    safety_data = load_json_list(path_safety)

    age_ids = {item.get("id", "") for item in age_data if isinstance(item, dict)}
    safety_ids = {
        item.get("id", "")
        for item in safety_data
        if isinstance(item, dict) and item.get("id", "").endswith("_0")
    }

    expected_safety_ids = {f"{age_id}_0" for age_id in age_ids if age_id}

    extra_in_safety = sorted(safety_ids - expected_safety_ids)
    missing_in_safety = sorted(expected_safety_ids - safety_ids)

    if extra_in_safety:
        print("Extra IDs in safety dataset not expected from age dataset:")
        for item in extra_in_safety:
            print(item)
    else:
        print("No extra safety IDs found.")

    if missing_in_safety:
        print("\nExpected safety IDs missing from safety dataset:")
        for item in missing_in_safety:
            print(item)
    else:
        print("No expected safety IDs are missing from safety dataset.")

    return extra_in_safety, missing_in_safety


path_safety = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/safety/all_data_cleaned.json"
path_age = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/age_rating/all_data_cleaned.json"
compare_age_and_safety_ids(path_age, path_safety)


In [ ]:
# Clean rationale formatting

import json
import re

# Delete markdown from the rationales.

def clean_rationale(text):
    if not isinstance(text, str):
        return text

    # Remove triple-backslash escaped quotes: \\\"text\\\" -> "text".
    text = text.replace('\\\\\\"', '"')

    # Remove double/single-backslash escaped quotes.
    text = text.replace('\\\\"', '"')
    text = text.replace('\\"', '"')

    # Remove markdown emphasis.
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"\*(.*?)\*", r"\1", text)

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_record(record: dict) -> dict:
    for turn in record.get("conversations", []):

        if turn.get("from") != "gpt":
            continue

        value = turn.get("value", "")

        parsed = json.loads(value)

        if "rationale" in parsed:
            parsed["rationale"] = clean_rationale(
                parsed["rationale"]
            )

        turn["value"] = json.dumps(
            parsed,
            ensure_ascii=False,
            indent=4,
        )

    return record


In [ ]:
# Save cleaned rationales

import json

file = "/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/data/datasets/PEGI-Guard-DS/v4/age_rating/all_data.json"

with open(file, "r", encoding="utf-8") as f:
    data = json.load(f)

data = [clean_record(record) for record in data]

count = 0

for record in data:
    for turn in record.get("conversations", []):
        if turn.get("from") == "gpt":
            parsed = json.loads(turn["value"])
            rationale = parsed.get("rationale", "")

            if r'\\\"' in rationale:
                count += 1

print(count)

with open(file, "w", encoding="utf-8") as f:
    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved cleaned dataset to {file}")


In [ ]:
# Correct legacy UnsafeBench IDs

def fix_id(record):
    id_ = record.get("id", "")

    if id_.startswith("unsafebench"):
        # Unsafebench_1880_0_2 → unsafebench_1880_2.
        

        # Unsafebench_1880_0 → unsafebench_1880.
        id_ = re.sub(r"^(unsafebench_\d+)_0$", r"\1", id_)

        record["id"] = id_

    return record
